In [ ]:
from langgraph.constants import START, END
from typing import Literal
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph import MessagesState, StateGraph

from app.core.config import settings
from langchain.chat_models import init_chat_model

# 定义模型
model = init_chat_model(
    model="deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

# 1.定义状态
class IntentState(MessagesState):
    intent:str

# 定义节点
# 定义意图识别节点
def intent_node(state:IntentState):
    response:AIMessage = model.invoke(f"""
        根据用户的问题，进行意图识别。
        可选的结果：chat、weather、translation
        用户问题:{state["messages"][-1].content}
        """)
    return {"intent": response.text}

# 定义闲聊节点
def chat_node(state:IntentState):
    response:AIMessage = model.invoke(state["messages"])
    return {"messages":[response]}
# 定义查询天气节点
def weather_node(state: IntentState):
    return {
        "messages":[AIMessage(content="晴空万里~")]
    }
# 定义翻译节点
def translation_node(state: IntentState):
    response:AIMessage = model.invoke(f"""
    将用户的问题进行翻译，如果是中文翻译成英文，如果是英文翻译成中文。
    用户问题:{state["messages"][-1].content}
    """)
    return {
        "messages":[response]
    }

# 定义路由函数
# 3.定义路由函数
def router(state:IntentState) -> Literal["chat", "weather", "translation"]:
    return state["intent"]

# 4.定义边，构建智能体
builder = StateGraph(IntentState)

builder.add_node("intent", intent_node)
builder.add_node("chat", chat_node)
builder.add_node("weather", weather_node)
builder.add_node("translation", translation_node)

builder.add_edge(START, "intent")
builder.add_conditional_edges("intent", router)
builder.add_edge("chat", END)
builder.add_edge("weather", END)
builder.add_edge("translation")

agent = builder.compile()


# 5.调用智能体
result = agent.invoke({
    "messages":[
        # HumanMessage(content="今天北京的天气怎么样？")
        # HumanMessage(content="翻译:我爱睡觉~")
        HumanMessage(content="你好吗？")
    ]
})

print(f"路由识别结果:{result["intent"]}")
for message in result["messages"]:
    message.pretty_print()